<a href="https://colab.research.google.com/github/Datadog-995/Quality-Data-Solutions-Ecommerce-Audit-/blob/main/_Quality_Data_Solutions_Ecommerce_Audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
import pandas as pd

# Load the audited CSV dataset directly from your Colab workspace
file_path = '/content/Sheet 1-1-1-Ecommerce-Order-Data-Cleaning 3-Sample-2-copy-xlsx.csv'
df = pd.read_csv(file_path)

# Verify the shape and columns are correctly loaded
print(f"Dataset successfully loaded! Total rows: {df.shape[0]}, Total columns: {df.shape[1]}")
df.head(25)

Dataset successfully loaded! Total rows: 906, Total columns: 16


,Order ID,SKU,Product Name,Category,Price,Price_Audit_Flag,Quantity,Quantity_Audit_Flag,Payment Method,Order Status,Order Date,Customer Email,Data Quality Status,Audit Notes,Ecommerce Order Data - Audit Summary,Column 2
0,ORD-10001,SKU-1003,Stainless Steel Knife Set,Sports & Outdoors,NaN,NaN,1.0,NaN,Apple Pay,unfulfilled,NaN,customer1@example.com,NaN,NaN,NaN,NaN
1,ORD-10002,SKU-1043,Running Shoes Men,Clothing,24.64,NaN,5.0,NaN,Gift Card,cancelled,2025-12-03,customer2@example.com,NaN,NaN,NaN,NaN
2,ORD-10003,SKU-1012,Portable Phone Charger,Home & Kitchen,44.04,NaN,2.0,NaN,Pay Pal,refunded,2026-11-23,customer3@example.com,NaN,NaN,NaN,NaN
3,ORD-10004,SKU-1004,Leather Wallet Men,Toys,44.57,NaN,2.0,NaN,Apple Pay,refunded,2026-04-05,customer4@example.com,NaN,NaN,NaN,NaN
4,ORD-10005,SKU-1087,Memory Foam Pillow,Beauty Products,175.03,NaN,1.0,NaN,Apple Pay,cancelled,13/45/2026,customer5@example.com,NaN,NaN,NaN,NaN
5,ORD-10006,SKU-1064,Wireless Mouse,Home & Kitchen,34.29,NaN,1.0,NaN,Gift Card,fulfilled,2026-04-02,customer6@example.com,NaN,NaN,NaN,NaN
6,ORD-10007,SKU-1008,Wireless Mouse,Toys,53.59,NaN,2.0,NaN,Gift Card,refunded,2025-04-08,customer7@example.com,NaN,NaN,NaN,NaN
7,ORD-10008,SKU-1009,Baby Onesie Pack,Home & Kitchen,96.43,NaN,5.0,NaN,Pay Pal,refunded,2025-02-22,customer8@example.com,NaN,NaN,NaN,NaN
8,ORD-10009,SKU-1083,Wool Winter Scarf,Home & Kitchen,50.77,NaN,2.0,NaN,Credit Card,fulfilled,2025-02-25,customer9@example.com,NaN,NaN,NaN,NaN
9,ORD-10010,SKU-1007,Ceramic Coffee Mug Set,Beauty Products,80.76,NaN,2.0,NaN,Pay Pal,fulfilled,2025-01-19,customer10@example.com,NaN,NaN,NaN,NaN


In [18]:
import pandas as pd
import numpy as np

# 1. Correct "Pay Pal" to "PayPal"
df['Payment Method'] = df['Payment Method'].replace('Pay Pal', 'PayPal')

# 2. Flag incorrect dates in a new column right next to 'Order Date'
# Identify rows where a date exists but cannot be parsed correctly (e.g., '13/45/2026')
parsed_dates = pd.to_datetime(df['Order Date'], errors='coerce')
invalid_date_mask = df['Order Date'].notnull() & parsed_dates.isnull()

# Create the flag column
date_flags = np.where(invalid_date_mask, 'Invalid Date', 'Valid')

# Find the exact position of 'Order Date' and insert the flag column right next to it
if 'Order Date Audit Flag' in df.columns:
    df.drop(columns=['Order Date Audit Flag'], inplace=True)
order_date_idx = df.columns.get_loc('Order Date')
df.insert(order_date_idx + 1, 'Order Date Audit Flag', date_flags)

# 3. Fill missing prices by looking up the product's price from other rows using SKU
sku_price_mapping = df.dropna(subset=['Price']).groupby('SKU')['Price'].first()
df['Price'] = df['Price'].fillna(df['SKU'].map(sku_price_mapping))

# 4. Drop the empty audit and redundant columns to clean up the workspace
columns_to_drop = [
    'Price_Audit_Flag',
    'Quantity_Audit_Flag',
    'Data Quality Status',
    'Audit Notes',
    'Ecommerce Order Data - Audit Summary',
    'Column 2'
]
# Drop only columns that currently exist to prevent errors
existing_drops = [col for col in columns_to_drop if col in df.columns]
df.drop(columns=existing_drops, inplace=True)

# 5. Display the cleaned dataframe update
print("Audit processing and cleanup complete!")
print(f"Active columns remaining: {list(df.columns)}")
df.head(25)

Audit processing and cleanup complete!
Active columns remaining: ['Order ID', 'SKU', 'Product Name', 'Category', 'Price', 'Quantity', 'Payment Method', 'Order Status', 'Order Date', 'Order Date Audit Flag', 'Customer Email']


,Order ID,SKU,Product Name,Category,Price,Quantity,Payment Method,Order Status,Order Date,Order Date Audit Flag,Customer Email
0,ORD-10001,SKU-1003,Stainless Steel Knife Set,Sports & Outdoors,56.13,1.0,Apple Pay,unfulfilled,NaN,Valid,customer1@example.com
1,ORD-10002,SKU-1043,Running Shoes Men,Clothing,24.64,5.0,Gift Card,cancelled,2025-12-03,Valid,customer2@example.com
2,ORD-10003,SKU-1012,Portable Phone Charger,Home & Kitchen,44.04,2.0,PayPal,refunded,2026-11-23,Valid,customer3@example.com
3,ORD-10004,SKU-1004,Leather Wallet Men,Toys,44.57,2.0,Apple Pay,refunded,2026-04-05,Valid,customer4@example.com
4,ORD-10005,SKU-1087,Memory Foam Pillow,Beauty Products,175.03,1.0,Apple Pay,cancelled,13/45/2026,Invalid Date,customer5@example.com
5,ORD-10006,SKU-1064,Wireless Mouse,Home & Kitchen,34.29,1.0,Gift Card,fulfilled,2026-04-02,Valid,customer6@example.com
6,ORD-10007,SKU-1008,Wireless Mouse,Toys,53.59,2.0,Gift Card,refunded,2025-04-08,Valid,customer7@example.com
7,ORD-10008,SKU-1009,Baby Onesie Pack,Home & Kitchen,96.43,5.0,PayPal,refunded,2025-02-22,Valid,customer8@example.com
8,ORD-10009,SKU-1083,Wool Winter Scarf,Home & Kitchen,50.77,2.0,Credit Card,fulfilled,2025-02-25,Valid,customer9@example.com
9,ORD-10010,SKU-1007,Ceramic Coffee Mug Set,Beauty Products,80.76,2.0,PayPal,fulfilled,2025-01-19,Valid,customer10@example.com


In [19]:
import pandas as pd
import numpy as np

# 1. Correct "Pay Pal" to "PayPal"
df['Payment Method'] = df['Payment Method'].replace('Pay Pal', 'PayPal')

# 2. Flag incorrect dates in a new column right next to 'Order Date'
# Identify rows where a date exists but cannot be parsed correctly (e.g., '13/45/2026')
parsed_dates = pd.to_datetime(df['Order Date'], errors='coerce')
invalid_date_mask = df['Order Date'].notnull() & parsed_dates.isnull()

# Create the flag column
date_flags = np.where(invalid_date_mask, 'Invalid Date', 'Valid')

# Find the exact position of 'Order Date' and insert the flag column right next to it
if 'Order Date Audit Flag' in df.columns:
    df.drop(columns=['Order Date Audit Flag'], inplace=True)
order_date_idx = df.columns.get_loc('Order Date')
df.insert(order_date_idx + 1, 'Order Date Audit Flag', date_flags)

# 3. Fill missing prices by looking up the product's price from other rows using SKU
sku_price_mapping = df.dropna(subset=['Price']).groupby('SKU')['Price'].first()
df['Price'] = df['Price'].fillna(df['SKU'].map(sku_price_mapping))

# 4. Drop the empty audit and redundant columns to clean up the workspace
columns_to_drop = [
    'Price_Audit_Flag',
    'Quantity_Audit_Flag',
    'Data Quality Status',
    'Audit Notes',
    'Ecommerce Order Data - Audit Summary',
    'Column 2'
]
# Drop only columns that currently exist to prevent errors
existing_drops = [col for col in columns_to_drop if col in df.columns]
df.drop(columns=existing_drops, inplace=True)

# 5. Display the cleaned dataframe update
print("Audit processing and cleanup complete!")
print(f"Active columns remaining: {list(df.columns)}")
df.head(25)

Audit processing and cleanup complete!
Active columns remaining: ['Order ID', 'SKU', 'Product Name', 'Category', 'Price', 'Quantity', 'Payment Method', 'Order Status', 'Order Date', 'Order Date Audit Flag', 'Customer Email']


,Order ID,SKU,Product Name,Category,Price,Quantity,Payment Method,Order Status,Order Date,Order Date Audit Flag,Customer Email
0,ORD-10001,SKU-1003,Stainless Steel Knife Set,Sports & Outdoors,56.13,1.0,Apple Pay,unfulfilled,NaN,Valid,customer1@example.com
1,ORD-10002,SKU-1043,Running Shoes Men,Clothing,24.64,5.0,Gift Card,cancelled,2025-12-03,Valid,customer2@example.com
2,ORD-10003,SKU-1012,Portable Phone Charger,Home & Kitchen,44.04,2.0,PayPal,refunded,2026-11-23,Valid,customer3@example.com
3,ORD-10004,SKU-1004,Leather Wallet Men,Toys,44.57,2.0,Apple Pay,refunded,2026-04-05,Valid,customer4@example.com
4,ORD-10005,SKU-1087,Memory Foam Pillow,Beauty Products,175.03,1.0,Apple Pay,cancelled,13/45/2026,Invalid Date,customer5@example.com
5,ORD-10006,SKU-1064,Wireless Mouse,Home & Kitchen,34.29,1.0,Gift Card,fulfilled,2026-04-02,Valid,customer6@example.com
6,ORD-10007,SKU-1008,Wireless Mouse,Toys,53.59,2.0,Gift Card,refunded,2025-04-08,Valid,customer7@example.com
7,ORD-10008,SKU-1009,Baby Onesie Pack,Home & Kitchen,96.43,5.0,PayPal,refunded,2025-02-22,Valid,customer8@example.com
8,ORD-10009,SKU-1083,Wool Winter Scarf,Home & Kitchen,50.77,2.0,Credit Card,fulfilled,2025-02-25,Valid,customer9@example.com
9,ORD-10010,SKU-1007,Ceramic Coffee Mug Set,Beauty Products,80.76,2.0,PayPal,fulfilled,2025-01-19,Valid,customer10@example.com


In [21]:
import pandas as pd
import numpy as np
import re

# 1. Reload original or work with current state safely
# (Ensuring columns exist for the fresh audit run)
df_clean = df.copy()

# 2. Flag missing prices in a dedicated 'Price Audit Column' right next to Price
# If the original Price was NaN (before we mapped it), we flag it as 'Missing Price'
is_missing_price = df_clean['Price'].isnull()

# Fill the missing prices using the SKU mapping as before so your data stays complete
sku_price_mapping = df_clean.dropna(subset=['Price']).groupby('SKU')['Price'].first()
df_clean['Price'] = df_clean['Price'].fillna(df_clean['SKU'].map(sku_price_mapping))

# Create the Audit Column
price_flags = np.where(is_missing_price, 'Missing Price - Restored via SKU', 'Valid')

# Insert it directly to the right of the Price column
if 'Price Audit Column' in df_clean.columns:
    df_clean.drop(columns=['Price Audit Column'], inplace=True)
price_idx = df_clean.columns.get_loc('Price')
df_clean.insert(price_idx + 1, 'Price Audit Column', price_flags)

# 3. Keep your other advanced cleaning steps active
# Standardize Order Status to clean lowercase
if 'Order Status' in df_clean.columns:
    df_clean['Order Status'] = df_clean['Order Status'].astype(str).str.strip().str.lower()

# Cast Quantity cleanly to integers
if 'Quantity' in df_clean.columns:
    df_clean['Quantity'] = pd.to_numeric(df_clean['Quantity'], errors='coerce').fillna(0).astype(int)

# Validate Email Formats
def is_valid_email(email):
    if pd.isna(email): return False
    pattern = r'^[\w\.-]+@[\w\.-]+\.\w+$'
    return bool(re.match(pattern, str(email)))

if 'Customer Email' in df_clean.columns:
    df_clean['Email_Valid'] = df_clean['Customer Email'].apply(is_valid_email)

# Remove completely duplicate rows
df_clean = df_clean.drop_duplicates()

# Point your active dataframe to this fully optimized version
df = df_clean
print("Price auditing column successfully inserted!")
df.head(25)


Price auditing column successfully inserted!


,Order ID,SKU,Product Name,Category,Price,Price Audit Column,Quantity,Payment Method,Order Status,Order Date,Order Date Audit Flag,Customer Email,Email_Valid
0,ORD-10001,SKU-1003,Stainless Steel Knife Set,Sports & Outdoors,56.13,Valid,1,Apple Pay,unfulfilled,NaN,Valid,customer1@example.com,True
1,ORD-10002,SKU-1043,Running Shoes Men,Clothing,24.64,Valid,5,Gift Card,cancelled,2025-12-03,Valid,customer2@example.com,True
2,ORD-10003,SKU-1012,Portable Phone Charger,Home & Kitchen,44.04,Valid,2,PayPal,refunded,2026-11-23,Valid,customer3@example.com,True
3,ORD-10004,SKU-1004,Leather Wallet Men,Toys,44.57,Valid,2,Apple Pay,refunded,2026-04-05,Valid,customer4@example.com,True
4,ORD-10005,SKU-1087,Memory Foam Pillow,Beauty Products,175.03,Valid,1,Apple Pay,cancelled,13/45/2026,Invalid Date,customer5@example.com,True
5,ORD-10006,SKU-1064,Wireless Mouse,Home & Kitchen,34.29,Valid,1,Gift Card,fulfilled,2026-04-02,Valid,customer6@example.com,True
6,ORD-10007,SKU-1008,Wireless Mouse,Toys,53.59,Valid,2,Gift Card,refunded,2025-04-08,Valid,customer7@example.com,True
7,ORD-10008,SKU-1009,Baby Onesie Pack,Home & Kitchen,96.43,Valid,5,PayPal,refunded,2025-02-22,Valid,customer8@example.com,True
8,ORD-10009,SKU-1083,Wool Winter Scarf,Home & Kitchen,50.77,Valid,2,Credit Card,fulfilled,2025-02-25,Valid,customer9@example.com,True
9,ORD-10010,SKU-1007,Ceramic Coffee Mug Set,Beauty Products,80.76,Valid,2,PayPal,fulfilled,2025-01-19,Valid,customer10@example.com,True


In [ ]:
import numpy as np
import pandas as pd

# Recalculate parsed dates to accurately catch completely empty values
parsed_dates = pd.to_datetime(df['Order Date'], errors='coerce')

# Flag if it is a corrupted string OR if it is completely blank (null)
date_conditions = [
    (df['Order Date'].notnull() & parsed_dates.isnull()), # Corrupted string
    (df['Order Date'].isnull())                          # Completely blank
]
date_choices = ['Invalid Date String', 'Missing Date Entry']

# Update the Audit Flag column
df['Order Date Audit Flag'] = np.select(date_conditions, date_choices, default='Valid')

df.head(25)

In [24]:
import pandas as pd
import numpy as np
import re

# 1. Take a clean copy of the current dataframe state
df_final = df.copy()

# 2. Track original anomalies before making corrections
is_missing_price = df_final['Price'].isnull()
# Check for numbers below zero (or text indicators of previous negatives if applicable)
is_negative_price = pd.to_numeric(df_final['Price'], errors='coerce') < 0

# 3. Apply the Price Corrections
# Fill original missing cells via SKU lookup mapping
sku_price_mapping = df_final.dropna(subset=['Price']).groupby('SKU')['Price'].first()
df_final['Price'] = df_final['Price'].fillna(df_final['SKU'].map(sku_price_mapping))

# Convert all negative prices to clean, positive absolute values
df_final['Price'] = df_final['Price'].abs()

# 4. Generate the Comprehensive Price Audit Column
price_conditions = [is_missing_price, is_negative_price]
price_choices = ['Missing Price - Restored via SKU', 'Negative Price - Converted to Positive']
df_final['Price Audit Column'] = np.select(price_conditions, price_choices, default='Valid')

# 5. Comprehensive Date Audit (Catching strings vs missing blanks)
parsed_dates = pd.to_datetime(df_final['Order Date'], errors='coerce')
date_conditions = [
    (df_final['Order Date'].notnull() & parsed_dates.isnull()), # Bad formats
    (df_final['Order Date'].isnull())                           # Empty cells
]
date_choices = ['Invalid Date String', 'Missing Date Entry']
df_final['Order Date Audit Flag'] = np.select(date_conditions, date_choices, default='Valid')

# 6. Re-run structural foundations
if 'Order Status' in df_final.columns:
    df_final['Order Status'] = df_final['Order Status'].astype(str).str.strip().str.lower()

if 'Quantity' in df_final.columns:
    df_final['Quantity'] = pd.to_numeric(df_final['Quantity'], errors='coerce').fillna(0).astype(int)

# Bind the final audited dataset to the main reference environment
df = df_final
print("Audit pipeline successfully updated and executed!")
df.head(25)

Audit pipeline successfully updated and executed!


,Order ID,SKU,Product Name,Category,Price,Price Audit Column,Quantity,Payment Method,Order Status,Order Date,Order Date Audit Flag,Customer Email,Email_Valid
0,ORD-10001,SKU-1003,Stainless Steel Knife Set,Sports & Outdoors,56.13,Valid,1,Apple Pay,unfulfilled,NaN,Missing Date Entry,customer1@example.com,True
1,ORD-10002,SKU-1043,Running Shoes Men,Clothing,24.64,Valid,5,Gift Card,cancelled,2025-12-03,Valid,customer2@example.com,True
2,ORD-10003,SKU-1012,Portable Phone Charger,Home & Kitchen,44.04,Valid,2,PayPal,refunded,2026-11-23,Valid,customer3@example.com,True
3,ORD-10004,SKU-1004,Leather Wallet Men,Toys,44.57,Valid,2,Apple Pay,refunded,2026-04-05,Valid,customer4@example.com,True
4,ORD-10005,SKU-1087,Memory Foam Pillow,Beauty Products,175.03,Valid,1,Apple Pay,cancelled,13/45/2026,Invalid Date String,customer5@example.com,True
5,ORD-10006,SKU-1064,Wireless Mouse,Home & Kitchen,34.29,Valid,1,Gift Card,fulfilled,2026-04-02,Valid,customer6@example.com,True
6,ORD-10007,SKU-1008,Wireless Mouse,Toys,53.59,Valid,2,Gift Card,refunded,2025-04-08,Valid,customer7@example.com,True
7,ORD-10008,SKU-1009,Baby Onesie Pack,Home & Kitchen,96.43,Valid,5,PayPal,refunded,2025-02-22,Valid,customer8@example.com,True
8,ORD-10009,SKU-1083,Wool Winter Scarf,Home & Kitchen,50.77,Valid,2,Credit Card,fulfilled,2025-02-25,Valid,customer9@example.com,True
9,ORD-10010,SKU-1007,Ceramic Coffee Mug Set,Beauty Products,80.76,Valid,2,PayPal,fulfilled,2025-01-19,Valid,customer10@example.com,True


In [25]:
# Filter and display all rows that originally had negative prices
negative_rows = df[df['Price Audit Column'] == 'Negative Price - Converted to Positive']

print(f"Total rows found with negative prices: {len(negative_rows)}")
# Display key identifier columns along with the corrected positive Price
negative_rows[['Order ID', 'SKU', 'Product Name', 'Category', 'Price', 'Price Audit Column']]


Total rows found with negative prices: 16


,Order ID,SKU,Product Name,Category,Price,Price Audit Column
59,ORD-10060,SKU-1028,Kids Backpack,Clothing,94.35,Negative Price - Converted to Positive
74,ORD-10075,SKU-1072,Running Shoes Women,Clothing,108.40,Negative Price - Converted to Positive
91,ORD-10092,SKU-1067,Stainless Steel Water Bottle,Apparel,78.93,Negative Price - Converted to Positive
187,ORD-10188,SKU-1084,Running Shoes Men,Electronics,25.72,Negative Price - Converted to Positive
218,ORD-10219,SKU-1015,Wireless Bluetooth Earbuds,Electronics,79.22,Negative Price - Converted to Positive
227,ORD-10228,SKU-1098,Kids Backpack,Apparel,148.53,Negative Price - Converted to Positive
288,ORD-10289,SKU-1044,Organic Cotton T-shirt,Sports & Outdoors,104.88,Negative Price - Converted to Positive
300,ORD-10301,SKU-1064,Kids Puzzle 100pc,Home & Kitchen,29.15,Negative Price - Converted to Positive
309,ORD-10310,SKU-1090,Running Shoes Women,Electronics,172.65,Negative Price - Converted to Positive
330,ORD-10331,SKU-1024,Memory Foam Pillow,Beauty Products,72.45,Negative Price - Converted to Positive
